In [1]:
%pip install langchain-ollama langchain langchain-community pydantic>=2.0 --upgrade

zsh:1: 2.0 not found
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="mistral", temperature=0.0)

In [10]:
# %pip install langchain
# %pip install langchain-community
%pip install docarray

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
# Modern LangChain imports
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from IPython.display import display, Markdown

file = "data.csv"
loader = CSVLoader(file_path=file)
documents = loader.load()


In [4]:
documents[0]

Document(metadata={'source': 'data.csv', 'row': 0}, page_content='product: Wireless Headphones\nreview: Amazing sound quality! Battery lasts all day and the noise cancellation works great. Highly recommend for music lovers.')

In [8]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
embed = embeddings.embed_query("Hi, I am trying to study Agentic AI")

In [13]:
print(len(embed))
print(embed[:5])

768
[-0.056607574, 0.03702436, -0.11754525, 0.024928043, 0.04979292]


In [12]:
db = DocArrayInMemorySearch.from_documents(documents, embeddings)

In [14]:
query = "Please suggest a gadget that has good reviews"
docs1 = db.similarity_search(query)

In [16]:
list(docs1)

[Document(metadata={'source': 'data.csv', 'row': 0}, page_content='product: Wireless Headphones\nreview: Amazing sound quality! Battery lasts all day and the noise cancellation works great. Highly recommend for music lovers.'),
 Document(metadata={'source': 'data.csv', 'row': 1}, page_content='product: Smart Watch\nreview: Tracks my fitness perfectly and syncs with my phone seamlessly. The heart rate monitor is accurate and the display is bright.'),
 Document(metadata={'source': 'data.csv', 'row': 10}, page_content='product: Desk Lamp\nreview: Adjustable brightness levels are perfect. USB charging port is a nice bonus. Modern design looks great on my desk.'),
 Document(metadata={'source': 'data.csv', 'row': 3}, page_content="product: Coffee Maker\nreview: Makes perfect coffee every morning. The programmable timer is convenient and it's easy to clean. Love it!")]

In [15]:
docs1[0]

Document(metadata={'source': 'data.csv', 'row': 0}, page_content='product: Wireless Headphones\nreview: Amazing sound quality! Battery lasts all day and the noise cancellation works great. Highly recommend for music lovers.')

In [18]:
qdocs1 = "".join(docs1[i].page_content for i in range(len(docs1)))

In [21]:
response = llm.invoke(f"{qdocs1} Question: Please list all the gadgets \
with a good review in a table in markdown and summarize each one.")

In [24]:
display(Markdown(response.content))

Here is the table of gadgets with good reviews:

| Product | Review Summary |
| --- | --- |
| Wireless Headphones | Excellent sound quality, long battery life, and effective noise cancellation. Ideal for music enthusiasts. |
| Smart Watch | Accurate fitness tracking, seamless phone syncing, and a bright display. Great for those who want to stay connected and on top of their health. |
| Desk Lamp | Adjustable brightness levels, convenient USB charging port, and a modern design that looks great on any desk. Perfect for task lighting or ambient illumination. |
| Coffee Maker | Consistently produces perfect coffee with ease, thanks to the programmable timer and easy cleaning process. A must-have for coffee lovers. |

Let me know if you'd like me to add anything else!

In [27]:
retriever = db.as_retriever()

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
{context}

Question: {question}
"""
)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    |prompt
    | llm
    | StrOutputParser()
)

In [29]:
response = qa_chain.invoke("Please list all the gadgets with good reviews in a table \
in markdown and summarize each one.")

In [30]:
display(Markdown(response))

| Product | Review Summary |
| --- | --- |
| Desk Lamp | Adjustable brightness levels, USB charging port, modern design |
| Wireless Headphones | Amazing sound quality, long battery life, effective noise cancellation |
| Smart Watch | Tracks fitness perfectly, accurate heart rate monitor, bright display |
| Backpack | Durable material, multiple compartments, well-padded laptop sleeve |

Note: The summaries are based on the provided reviews and may not be exhaustive.

In [31]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown

# This block replaces: VectorstoreIndexCreator().from_loaders([loader])
llm = ChatOllama(model="mistral", temperature=0.0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
db = DocArrayInMemorySearch.from_documents(
    CSVLoader("data.csv").load(), embeddings
)

# This block replaces: index.query("question", llm=llm)
qa_chain = (
    {
        "context": db.as_retriever() | (lambda docs: "\n\n".join(d.page_content for d in docs)),
        "question": RunnablePassthrough()
    }
    | ChatPromptTemplate.from_template("Context:\n{context}\n\nQuestion: {question}")
    | llm
    | StrOutputParser()
)

In [ ]:
display(Markdown(qa_chain.invoke("List all gadgets with good reviews in a markdown table and summarize each one.")))

In [32]:
examples = [
    {
        "question": "Does Smart Watch track heart rate?",
        "answer": "Yes"
    },
    {
        "question": "What product is good for allergies?",
        "answer": "Air Purifier"
    }
]

In [33]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

qa_generation_prompt = ChatPromptTemplate.from_template(
    """Given the following document, generate a question and answer pair.
The question should be answerable from the document.
Document: {doc}

Return ONLY valid JSON with keys "question" and "answer". No extra text before or after the JSON.""")

qa_gen_chain = qa_generation_prompt | llm | JsonOutputParser()

qa_pairs = []
for doc in documents:
    qa_pairs.append(qa_gen_chain.invoke({"doc": doc.page_content}))
display(qa_pairs)


[{'question': 'What are some positive aspects of the Wireless Headphones mentioned in the review?',
  'answer': 'Amazing sound quality, battery lasts all day, and noise cancellation works great.'},
 {'question': 'What are some positive aspects of the Smart Watch mentioned in the review?',
  'answer': 'The Smart Watch tracks fitness perfectly, syncs with the phone seamlessly, has an accurate heart rate monitor, and a bright display.'},
 {'question': 'What are some benefits of using the Laptop Stand mentioned in the review?',
  'answer': 'The Laptop Stand improves posture significantly, has a sturdy aluminum build, and is adjustable in height.'},
 {'question': 'What does the reviewer think about the Coffee Maker?',
  'answer': 'The reviewer loves the Coffee Maker as it makes perfect coffee every morning, has a convenient programmable timer, and is easy to clean.'},
 {'question': 'What are some positive aspects of the Yoga Mat mentioned in the review?',
  'answer': 'The Yoga Mat has excel

In [34]:
examples += qa_pairs

In [35]:
qa_chain.invoke(examples[0]["question"])

' Yes, according to the review provided, the Smart Watch does track heart rate accurately.'

In [36]:
predictions = qa_chain.batch([ex["question"] for ex in examples])

In [16]:
display(predictions)

['Yes, according to the review, the Smart Watch tracks fitness perfectly and has an accurate heart rate monitor.',
 'The Air Purifier is the product that is good for allergies, according to the review. It mentions that it "Noticed cleaner air within days" and states that it\'s "Great for allergies".',
 'Based on the context, it appears that this product is a pair of Wireless Headphones.',
 'Yes, according to the review, the Smart Watch tracks your fitness perfectly.',
 'The laptop stand is made of sturdy aluminum.',
 'There is no mention of a Coffee Maker in the provided reviews. The reviews are for Electric Toothbrush, Wireless Headphones, Desk Lamp, and Air Purifier.',
 'The benefits of this yoga mat include:\n\n1. Excellent grip to prevent slipping or sliding during practice.\n2. Good cushioning for comfort and support, particularly for knees.\n3. Non-slip surface that works well even when sweaty, providing stability and security.\n\nThese features suggest that the yoga mat is desig

In [37]:
from langchain_classic.evaluation import QAEvalChain

eval_chain = QAEvalChain.from_llm(llm)

In [38]:
graded_output = eval_chain.evaluate(
    examples,
    [{"query": ex["question"], "result": pred} for ex, pred in zip(examples, predictions)],
    question_key="question",
    answer_key="answer",
    prediction_key="result"
)

In [19]:
display(graded_output)

[{'results': 'INCORRECT\n\nThe student\'s answer contains additional information that is not present in the true answer. The true answer only states "Yes", without any additional details about the accuracy of the heart rate monitor.'},
 {'results': 'INCORRECT\n\nThe student answer provides additional information about the product being reviewed, but the true answer is simply "Air Purifier", which does not include any specific details or reviews.'},
 {'results': 'INCORRECT\n\nThe student\'s answer mentions "Wireless Headphones", which includes additional information not present in the true answer. The correct answer only states "Wireless".'},
 {'results': 'INCORRECT\n\nThe student\'s answer contains additional information ("according to the review", "tracks your fitness perfectly") that is not present in the true answer. The factual accuracy of the student\'s answer is only based on the single word "Yes".'},
 {'results': 'INCORRECT\n\nThe student\'s answer mentions "sturdy", which is no

In [39]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print(f"Question: {eg['question']}")
    print(f"Real Answer: {eg['answer']}")
    print(f"Predicted Answer: {predictions[i]}")
    print(f"Predicted Grade: {graded_output[i]['results']}")

Example 0:
Question: Does Smart Watch track heart rate?
Real Answer: Yes
Predicted Answer:  Yes, according to the review provided, the Smart Watch does track heart rate accurately.
Predicted Grade:  CORRECT
Example 1:
Question: What product is good for allergies?
Real Answer: Air Purifier
Predicted Answer:  Based on the reviews provided, the Air Purifier seems to be a product that is good for allergies as it noticeably improves air quality within days, which could be beneficial for individuals with allergies.
Predicted Grade:  CORRECT
Example 2:
Question: What are some positive aspects of the Wireless Headphones mentioned in the review?
Real Answer: Amazing sound quality, battery lasts all day, and noise cancellation works great.
Predicted Answer:  The positive aspects of the Wireless Headphones, as mentioned in the review, include:
1. Amazing sound quality
2. Battery that lasts all day
3. Effective noise cancellation
4. Highly recommended for music lovers
Predicted Grade:  CORRECT
Exa